# 01 — Correlation Engine

**Purpose:** Build rolling correlations across the full asset universe, score pair stability, detect breakdowns, compute tradeability scores, and run lead-lag analysis. Export results to `research/outputs/correlation_state.json` for the dashboard.

**Run time:** ~2–4 minutes on the full 100+ ticker universe (dominated by yfinance fetch).

**Output:** `research/outputs/correlation_state.json`

---
**Sections:**
1. Setup & data fetch
2. Rolling correlation matrices (30d / 90d / 180d)
3. Stability scoring
4. Breakdown detection
5. Tradeability scoring
6. Lead-lag analysis (held tickers only)
7. Visualisations
8. Export to JSON

## 1. Setup & data fetch

In [1]:
import sys
import os
import json
import logging
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Path setup: allow imports from both research/src and portfolio/src ──
NOTEBOOK_DIR  = os.path.abspath('')
RESEARCH_ROOT = os.path.join(NOTEBOOK_DIR, '..')
PORTFOLIO_ROOT = os.path.join(RESEARCH_ROOT, '..', 'portfolio')

sys.path.insert(0, PORTFOLIO_ROOT)
sys.path.insert(0, RESEARCH_ROOT)

# ── Research imports ──
from src.config import (
    ASSET_UNIVERSE, BENCHMARK_TICKER, LOOKBACK_DAYS, RISK_FREE_RATE,
    ROLLING_WINDOWS, CORRELATION_BREAKDOWN_THRESHOLD, MIN_CORRELATION_THRESHOLD,
    MAX_STABILITY_STD, TOP_PAIRS_N, MAX_LAG_DAYS, OUTPUT_CORRELATION,
)
from src.correlation import (
    latest_correlation_matrix,
    compute_stability_scores,
    detect_correlation_breakdowns,
    compute_tradeability_scores,
    compute_lead_lag,
)

# ── Portfolio data layer (shared, read-only) ──
from src.data_loader import (
    fetch_historical,
    calculate_log_returns,
    fetch_fx_rate,
    convert_usd_prices_to_eur,
    load_ledger,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# ── Plotting style ──
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.size':        11,
})

print('✅ Imports OK')
print(f'   Asset universe : {len(ASSET_UNIVERSE)} tickers')
print(f'   Lookback       : {LOOKBACK_DAYS} trading days (~2 years)')
print(f'   Rolling windows: {ROLLING_WINDOWS} days')
print(f'   Output path    : {OUTPUT_CORRELATION}')

ImportError: cannot import name 'ROLLING_WINDOWS' from 'src.config' (c:\Users\ahmty\Desktop\hedge-fund\research\notebooks\..\..\portfolio\src\config.py)

In [ ]:
# ── Fetch prices (uses portfolio cache at portfolio/data/historical_prices.csv
#    if available, otherwise fetches fresh from yfinance) ──

CACHE_PATH = os.path.join(PORTFOLIO_ROOT, 'data', 'historical_prices.csv')

print('Fetching market data...')
prices_raw = fetch_historical(
    tickers     = ASSET_UNIVERSE,
    lookback_days = LOOKBACK_DAYS,
    cache_path  = CACHE_PATH,
)

# Convert USD tickers to EUR (same logic as recalculate_engine.py)
usd_eur = fetch_fx_rate('USD', 'EUR')
prices  = convert_usd_prices_to_eur(prices_raw, usd_eur)

# Log returns
log_returns = calculate_log_returns(prices)

print(f'\n✅ Data ready')
print(f'   Price shape    : {prices.shape}')
print(f'   Returns shape  : {log_returns.shape}')
print(f'   Date range     : {log_returns.index[0].date()} → {log_returns.index[-1].date()}')
print(f'   USD/EUR rate   : {usd_eur:.4f}')

In [ ]:
# ── Identify held tickers from ledger ──
# Lead-lag and some visualisations focus on held positions only.

LEDGER_PATH = os.path.join(PORTFOLIO_ROOT, 'data', 'ledger.csv')
holdings, cash = load_ledger(LEDGER_PATH)

# Filter to tickers that are both held and have return data
held_tickers = [t for t in holdings.keys() if t in log_returns.columns]

print(f'✅ Ledger loaded')
print(f'   Holdings       : {len(holdings)} tickers')
print(f'   Held + in data : {len(held_tickers)} tickers')
print(f'   Cash           : €{cash:,.2f}')
print(f'   Held tickers   : {held_tickers}')

## 2. Rolling correlation matrices

In [ ]:
# Compute latest correlation matrices at each rolling window
# (latest snapshot only — fast)

corr_matrices = {}
for w in ROLLING_WINDOWS:
    corr_matrices[w] = latest_correlation_matrix(log_returns, window=w)
    print(f'  Window {w:3d}d: correlation matrix {corr_matrices[w].shape}')

print('\n✅ Rolling correlation matrices computed')

In [ ]:
# ── Visualise: correlation heatmap for held tickers (90d window) ──

if len(held_tickers) >= 2:
    corr_held_90 = log_returns[held_tickers].tail(90).corr()

    fig, ax = plt.subplots(figsize=(max(8, len(held_tickers)), max(6, len(held_tickers) * 0.8)))
    mask = np.triu(np.ones_like(corr_held_90, dtype=bool), k=1)

    sns.heatmap(
        corr_held_90,
        ax       = ax,
        annot    = True,
        fmt      = '.2f',
        cmap     = 'RdYlGn',
        vmin     = -1,
        vmax     = 1,
        linewidths = 0.5,
        square   = True,
    )
    ax.set_title('Correlation heatmap — held tickers (90-day rolling)', pad=14, fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('⚠️  Fewer than 2 held tickers in data — skipping heatmap.')
    print('   (Run recalculate_engine.py first, or add entries to ledger.csv)')

## 3. Stability scoring

For every pair in the universe, measures the standard deviation of its 90-day rolling correlation over time.
Low std = stable relationship. High std = correlation flips around — dangerous for hedging.

In [ ]:
print('Computing stability scores (this takes ~30–60s for 100+ tickers)...')
stability_df = compute_stability_scores(log_returns, window=90)

print(f'\n✅ Stability scores computed for {len(stability_df)} pairs')
print(f'   Most stable (top 10):')
display(stability_df.head(10))

print(f'\n   Most unstable (bottom 10):')
display(stability_df.tail(10))

In [ ]:
# ── Visualise: distribution of stability scores ──

if not stability_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].hist(stability_df['stability_score'], bins=40, color='steelblue', edgecolor='white', linewidth=0.5)
    axes[0].set_title('Distribution of stability scores')
    axes[0].set_xlabel('Stability score (1 = perfectly stable)')
    axes[0].set_ylabel('Number of pairs')

    axes[1].hist(stability_df['mean_corr'], bins=40, color='coral', edgecolor='white', linewidth=0.5)
    axes[1].set_title('Distribution of mean correlation')
    axes[1].set_xlabel('Mean 90d correlation')
    axes[1].axvline(0, color='black', linestyle='--', linewidth=0.8)

    plt.tight_layout()
    plt.show()

## 4. Breakdown detection

Flags pairs where 30-day correlation has diverged from 90-day correlation by more than the threshold.
- **Compressing**: assets moving together more than usual → possible crisis signal
- **Decoupling**: relationship weakening → hedge may be breaking

In [ ]:
breakdown_alerts = detect_correlation_breakdowns(
    log_returns,
    short_window = 30,
    long_window  = 90,
    threshold    = CORRELATION_BREAKDOWN_THRESHOLD,
)

print(f'✅ Breakdown detection complete')
print(f'   Total alerts : {len(breakdown_alerts)}')

if breakdown_alerts:
    alerts_df = pd.DataFrame(breakdown_alerts)
    print(f'   High severity: {len(alerts_df[alerts_df.severity=="high"])} pairs')
    print(f'\n   Top 15 breakdown pairs:')
    display(alerts_df.head(15))
else:
    print('   No significant breakdowns detected at current threshold.')

## 5. Tradeability scoring

Combines correlation strength (0–4 pts), stability (0–3 pts), and volatility compatibility (0–3 pts)
into a single 0–10 score per pair. Tags each pair as `hedge`, `stat-arb`, or `monitor`.

In [ ]:
tradeability_df = compute_tradeability_scores(
    log_returns,
    stability_df,
    min_abs_corr = MIN_CORRELATION_THRESHOLD,
)

print(f'✅ Tradeability scoring complete')
print(f'   Pairs scored  : {len(tradeability_df)}')

if not tradeability_df.empty:
    tag_counts = tradeability_df['tag'].value_counts()
    for tag, count in tag_counts.items():
        print(f'   {tag:10s}: {count} pairs')

    print(f'\n   Top {TOP_PAIRS_N} tradeable pairs:')
    display(tradeability_df.head(TOP_PAIRS_N))

In [ ]:
# ── Visualise: top 20 pairs by tradeability score ──

if not tradeability_df.empty:
    top = tradeability_df.head(TOP_PAIRS_N).copy()
    top['pair'] = top['ticker_a'] + ' / ' + top['ticker_b']

    color_map = {'hedge': '#E07B54', 'stat-arb': '#4A90D9', 'monitor': '#888'}
    colors = top['tag'].map(color_map).tolist()

    fig, ax = plt.subplots(figsize=(12, 7))
    bars = ax.barh(top['pair'][::-1], top['tradeability'][::-1], color=colors[::-1], edgecolor='white')
    ax.set_xlabel('Tradeability score (max 10)')
    ax.set_title(f'Top {TOP_PAIRS_N} tradeable pairs')
    ax.set_xlim(0, 10)

    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=v, label=k) for k, v in color_map.items()]
    ax.legend(handles=legend_elements, loc='lower right')

    plt.tight_layout()
    plt.show()

## 6. Lead-lag analysis (held tickers)

Runs cross-correlation analysis on held tickers only (fast).
Identifies which assets move first — actionable alpha signal, not just hedging.

In [ ]:
lead_lag_results = []

if len(held_tickers) >= 2:
    print(f'Running lead-lag on {len(held_tickers)} held tickers (max lag = {MAX_LAG_DAYS}d)...')
    lead_lag_results = compute_lead_lag(log_returns, held_tickers, max_lag=MAX_LAG_DAYS)

    print(f'\n✅ Lead-lag analysis complete')
    print(f'   Significant pairs: {len(lead_lag_results)}')

    if lead_lag_results:
        ll_df = pd.DataFrame(lead_lag_results)
        display(ll_df)
    else:
        print('   No significant lead-lag relationships found among held tickers.')
else:
    print('⚠️  Need at least 2 held tickers for lead-lag analysis.')
    print('   Skipping — will run on full universe in 05_lead_lag.ipynb')

## 7. Rolling correlation of held pairs over time

In [ ]:
# ── Visualise: rolling 90-day correlation chart for top 3 tradeable pairs ──

if not tradeability_df.empty:
    top3 = tradeability_df.head(3)

    fig, axes = plt.subplots(len(top3), 1, figsize=(14, 4 * len(top3)), sharex=True)
    if len(top3) == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, top3.iterrows()):
        a, b = row['ticker_a'], row['ticker_b']
        pair_returns = log_returns[[a, b]].dropna()
        rolling_corr = pair_returns[a].rolling(90).corr(pair_returns[b])

        ax.plot(rolling_corr.index, rolling_corr.values, linewidth=1.2, color='steelblue')
        ax.axhline(0, color='black', linestyle='--', linewidth=0.7, alpha=0.5)
        ax.axhline(row['mean_corr'], color='coral', linestyle='--', linewidth=0.8,
                   label=f'Mean = {row["mean_corr"]:.2f}')
        ax.fill_between(rolling_corr.index, rolling_corr.values, alpha=0.08, color='steelblue')
        ax.set_ylabel('Correlation')
        ax.set_ylim(-1, 1)
        ax.set_title(f'{a} / {b}  |  tag: {row["tag"]}  |  tradeability: {row["tradeability"]:.1f}/10')
        ax.legend(fontsize=9)

    axes[-1].set_xlabel('Date')
    plt.suptitle('Rolling 90-day correlation — top 3 pairs', y=1.01, fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('No tradeability data to plot yet.')

## 8. Export to JSON

Writes `research/outputs/correlation_state.json` — the file read by the Flask server's
`/data/research_state.json` endpoint and rendered in the dashboard Research tab.

In [ ]:
def df_to_records(df: pd.DataFrame, n: int = None) -> list:
    """Convert DataFrame to list of dicts, replacing NaN with None for JSON safety."""
    if df is None or df.empty:
        return []
    subset = df.head(n) if n else df
    return subset.where(pd.notna(subset), other=None).to_dict(orient='records')


# Build the JSON payload
state = {
    'generated_at':     datetime.now().isoformat(),
    'data_end_date':    str(log_returns.index[-1].date()),
    'n_tickers':        len(log_returns.columns),
    'held_tickers':     held_tickers,

    # Latest snapshot correlation matrices (held tickers only for dashboard)
    'corr_30d':         corr_matrices[30].loc[
                            [t for t in held_tickers if t in corr_matrices[30].index],
                            [t for t in held_tickers if t in corr_matrices[30].columns]
                        ].round(4).to_dict() if held_tickers else {},
    'corr_90d':         corr_matrices[90].loc[
                            [t for t in held_tickers if t in corr_matrices[90].index],
                            [t for t in held_tickers if t in corr_matrices[90].columns]
                        ].round(4).to_dict() if held_tickers else {},
    'corr_180d':        corr_matrices[180].loc[
                            [t for t in held_tickers if t in corr_matrices[180].index],
                            [t for t in held_tickers if t in corr_matrices[180].columns]
                        ].round(4).to_dict() if held_tickers else {},

    # Top tradeable pairs
    'top_pairs':        df_to_records(tradeability_df, n=TOP_PAIRS_N),

    # Breakdown alerts
    'breakdown_alerts': breakdown_alerts,

    # Lead-lag (held tickers)
    'lead_lag_pairs':   lead_lag_results,

    # Summary stats
    'summary': {
        'n_pairs_scored':       len(tradeability_df) if not tradeability_df.empty else 0,
        'n_hedge_pairs':        int((tradeability_df['tag'] == 'hedge').sum()) if not tradeability_df.empty else 0,
        'n_stat_arb_pairs':     int((tradeability_df['tag'] == 'stat-arb').sum()) if not tradeability_df.empty else 0,
        'n_breakdown_alerts':   len(breakdown_alerts),
        'n_lead_lag_pairs':     len(lead_lag_results),
        'top_pair':             f"{tradeability_df.iloc[0]['ticker_a']} / {tradeability_df.iloc[0]['ticker_b']}" if not tradeability_df.empty else None,
        'top_pair_score':       float(tradeability_df.iloc[0]['tradeability']) if not tradeability_df.empty else None,
    }
}

# Write output
os.makedirs(os.path.dirname(OUTPUT_CORRELATION), exist_ok=True)
with open(OUTPUT_CORRELATION, 'w', encoding='utf-8') as f:
    json.dump(state, f, indent=2, default=str)

print(f'✅ Exported to: {OUTPUT_CORRELATION}')
print(f'   File size: {os.path.getsize(OUTPUT_CORRELATION) / 1024:.1f} KB')
print(f'\nSummary:')
for k, v in state['summary'].items():
    print(f'   {k:30s}: {v}')

---
## Done

`correlation_state.json` is now ready. Next steps:
- Add the `/data/research_state.json` endpoint to `portfolio/server.py`
- Add the Research Signals tab to `portfolio/dashboard.html`
- Run `02_regime_detection.ipynb` to populate `regime_state.json`